In [1]:
# Get raw data
with open('input/24.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
# Part 1
raw_init, raw_gates = [i for i in rawinput.split('\n\n')]
init_val = {j[0]:int(j[1]) 
            for i in raw_init.split('\n') 
            for j in [i.split(': ')]}
gates = {j[1]:j[0].split() 
         for i in raw_gates.split('\n')
         for j in [i.split(' -> ')]}

def wire_value(wire,known):
    if wire not in known:
        if wire in init_val:
            known[wire] = init_val[wire]
        else:
            v1,op,v2 = gates[wire]
            known[wire] = eval(''.join([str(wire_value(v1,known)),
                                        {'AND':'&','OR':'|','XOR':'^'}[op],
                                        str(wire_value(v2,known))]))
    return known[wire]

def output_value():
    known = {}
    return int(''.join([str(wire_value(j,known)) 
                        for j in sorted([i 
                                         for i in gates 
                                         if i[0]=='z'])[::-1]]), 
               2)
    
output_value()

56939028423824

In [ ]:
# Part 2
def af_inner(wire,actform):
    if wire not in actform:
        if wire in init_val:
            actform[wire] = wire
        else:
            v1,op,v2 = gates[wire]
            actform[wire] = sorted([f'({af_inner(v1,actform)} {op}'
                                    +f' {af_inner(v2,actform)})',
                                    f'({af_inner(v2,actform)} {op}'
                                    +f' {af_inner(v1,actform)})'])[0]
    return actform[wire]

def get_actual_formulas(gates):
    actform = dict()
    for i in gates:
        af_inner(i,actform)
    return actform, {v:k for k,v in actform.items()}
        
def get_correct_formula(wire):
    wn = int(wire[1:])
    if wn:
        formstr = (f'(C{wn-1:02d} XOR (x{wn:02d} XOR y{wn:02d}))'
                   if f'x{wn:02d}' in init_val
                   else f'C{wn-1:02d}')
        for i in range(wn-1,-1,-1):
            replstr = (f'((C{i-1:02d} AND (x{i:02d} XOR y{i:02d})) OR (x{i:02d} AND y{i:02d}))'
                       if i else '(x00 AND y00)')
            formstr = formstr.replace(f'C{i:02d}', replstr)
        return formstr
    else:
        return '(x00 XOR y00)'
    
def split_terms(form):
    spl_ix = [0]+[i 
                  for i,j in enumerate(form[1:-1])
                  if (b:=(b if i else 0)+{'(':1,')':-1}.get(j,0))+(j==')')==0
                  and j==' ']+[len(form[1:-1])]
    return [form[1:-1][i:j].strip() 
            for i,j in zip(spl_ix[:-1],spl_ix[1:])]
    
def fix_swapped_wires(w,cform):
    aform = actform[w]
    if (at:=split_terms(aform)) == (ct:=split_terms(cform)):
        return []
    elif at[1]==ct[1] and len(vcomm:={*at[::2]}&{*ct[::2]})>0:
        return fix_swapped_wires([i for i in gates[w][::2] 
                                  if actform[i] not in vcomm][0],
                                 [*{*ct[::2]}-vcomm][0])
    else:
        gates[w], gates[actform_r[cform]] = gates[actform_r[cform]], gates[w]
        return [w, actform_r[cform]]

swapped = []
for w in sorted([i for i in gates if i[0]=='z']):
    actform, actform_r = get_actual_formulas(gates)
    swapped += fix_swapped_wires(w,get_correct_formula(w))
print(','.join(sorted(swapped)))

frn,gmq,vtj,wnf,wtt,z05,z21,z39
